# KLSE Index Movement Classification Using Long Short-Term Memory (LSTM)

In this notebook, we will download historical stock price data for companies listed on the Kuala Lumpur Stock Exchange (KLSE) using the `yfinance` library. We will then save this data to a CSV file for further analysis and modeling.

Import the necessary libraries and define the list of KLSE stock tickers that you want to download data for. You can modify the list based on your needs. In this example, we will download data for the KLSE index itself.

In [ ]:
import yfinance as yf
import pandas as pd
import matplotlib.pyplot as plt

Define the KLSE ticker. In this example, we will download data for the KLSE index itself, which is represented by the ticker "^KLSE". You can modify this to include specific stock tickers if you want to download data for individual companies.

In [ ]:
# Define the KLSE ticker
ticker = "^KLSE"

download historical stock price data for the KLSE index from Yahoo Finance. We will specify the date range from January 1, 2005, to January 1, 2026, and set the interval to daily data.

In [ ]:
# Download historical data for the KLSE index
df = yf.download(
    ticker,
    start="2005-01-01",
    end="2026-01-01",
    interval="1d"
)

Inspect the downloaded data to ensure it has been retrieved correctly. We will display the first few rows, check the dataset information, and look for any missing values.

In [ ]:
# Display first 5 rows
print(df.head())

In [ ]:
# Display last 5 rows
print(df.tail())

In [ ]:
# Check dataset size
print("Dataset shape:", df.shape)

In [ ]:
# Check column names
print("Columns:")
print(df.columns)

In [ ]:
# Check dataset information
print("\nDataset information:")
print(df.info())

In [ ]:
# Check missing values
print("\nMissing values:")
print(df.isnull().sum())

In [ ]:
# Show basic statistical summary
print("\nStatistical summary:")
print(df.describe())

Since there are no missing values in the dataset, we can proceed to save it to a CSV file for future use. We will name the file "KLSE_20_years_data.csv".

In [ ]:
# Save the dataset to a CSV file
df.to_csv("./Dataset/KLSE_20_years_data.csv")

print("Dataset saved successfully as KLSE_20_years_data.csv")

Finally, let's visualize the closing price of the KLSE index over time to get an overview of its movement during the 20-year period.

In [ ]:
# Plot the closing price over time
plt.figure(figsize=(12, 6))
plt.plot(df.index, df['Close'], label='KLSE Closing Price')
plt.title('KLSE Closing Price Over Time')
plt.xlabel('Date')
plt.ylabel('Closing Price (MYR)')
plt.legend()
plt.grid()
plt.show()

Now that we have the historical stock price data for the KLSE index, we can proceed to create a target variable for our classification task. The target variable will indicate whether the next trading day's closing price is higher than the current day's closing price (1) or not (0). We will also remove the last row of the dataset since it does not have a next trading day closing price.

In [ ]:
# Create target variable
# 1 = Next trading day closing price is higher than current day closing price
# 0 = Next trading day closing price is lower than or equal to current day closing price

df["Target"] = (df["Close"].shift(-1) > df["Close"]).astype(int)

# Remove the last row because there is no next trading day closing price
df = df.dropna()

# Check target result
print(df[["Close", "Target"]].head())
print(df[["Close", "Target"]].tail())

# Check class distribution
print(df["Target"].value_counts())

The class distribution of the target variable is imbalanced, with more records indicating that the next trading day's closing price is higher than the current day's closing price (1) compared to records indicating that it is lower or unchanged (0). This imbalance may affect the performance of our classification model, and we may need to consider techniques to address it during model training.

In [ ]:
plt.figure(figsize=(6, 4))
df["Target"].value_counts().plot(kind="bar")
plt.title("KLSE Movement Class Distribution")
plt.xlabel("Class")
plt.ylabel("Number of Records")
plt.xticks([0, 1], ["Down/Unchanged", "Up"], rotation=0)
plt.show()

Next, we will select the input features for our LSTM model. We will use the "Open", "High", "Low", "Close", and "Volume" columns as our features (X) and the "Target" column as our target variable (y). We will also check the shape of the features and target variable to ensure they are correctly defined.

In [ ]:
# Select input features for LSTM model
features = ["Open", "High", "Low", "Close", "Volume"]

X = df[features]
y = df["Target"]

# Display selected features
print(X.head())
print(y.head())

# Check feature shape
print("Feature shape:", X.shape)
print("Target shape:", y.shape)

Before training our LSTM model, we need to normalize the input features to ensure that they are on a similar scale. This is important for the convergence of the model during training. We will use the `MinMaxScaler` from the `sklearn.preprocessing` module to scale the features to a range between 0 and 1. After scaling the features, we will check the first few rows of the scaled data and the shape of the scaled features to confirm that the normalization has been applied correctly.

In [ ]:
from sklearn.preprocessing import MinMaxScaler

# Normalise the input features into range 0 to 1
scaler = MinMaxScaler()

X_scaled = scaler.fit_transform(X)

# Check scaled data
print(X_scaled[:5])
print("Scaled feature shape:", X_scaled.shape)

create sequences of input features for the LSTM model. We will use the previous 60 trading days' data to predict the movement of the next trading day. The `create_sequences` function will generate the sequences of input features (X_seq) and the corresponding target variable (y_seq) for training the LSTM model. After creating the sequences, we will check the shapes of the resulting arrays to ensure they are correctly formatted for model training.

In [ ]:
import numpy as np

# Function to create LSTM sequences
def create_sequences(X, y, time_steps=60):
    Xs, ys = [], []

    for i in range(time_steps, len(X)):
        Xs.append(X[i-time_steps:i])
        ys.append(y.iloc[i])

    return np.array(Xs), np.array(ys)


# Use previous 60 trading days to predict next trading day movement
time_steps = 60

X_seq, y_seq = create_sequences(X_scaled, y, time_steps)

print("X sequence shape:", X_seq.shape)
print("y sequence shape:", y_seq.shape)

The resulting shapes of the input sequences (X_seq) and the target variable (y_seq) indicate that we have successfully created the sequences for our LSTM model. The shape of X_seq is (number of samples, time steps, number of features), and the shape of y_seq is (number of samples,), which is suitable for training a classification model. Now we can proceed to split the data into training and testing sets for model evaluation.

In [ ]:
# Split data into training and testing sets
# 80% training data, 20% testing data

train_size = int(len(X_seq) * 0.8)

X_train = X_seq[:train_size]
X_test = X_seq[train_size:]

y_train = y_seq[:train_size]
y_test = y_seq[train_size:]

# Check data shapes
print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("y_train shape:", y_train.shape)
print("y_test shape:", y_test.shape)

The training and testing sets have been successfully created, with 80% of the data allocated for training and 20% for testing. The shapes of the training and testing sets indicate that we have the correct number of samples, time steps, and features for our LSTM model. Now we can proceed to build and train the LSTM model for classifying the movement of the KLSE index.

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout

# Build LSTM model
model = Sequential()

# First LSTM layer
model.add(LSTM(
    units=64,
    return_sequences=True,
    input_shape=(X_train.shape[1], X_train.shape[2])
))
model.add(Dropout(0.2))

# Second LSTM layer
model.add(LSTM(
    units=32,
    return_sequences=False
))
model.add(Dropout(0.2))

# Output layer for binary classification
model.add(Dense(1, activation="sigmoid"))

# Display model architecture
model.summary()

The LSTM model has been successfully built with two LSTM layers and a dense output layer for binary classification. The first LSTM layer has 64 units and returns sequences, while the second LSTM layer has 32 units and does not return sequences. Dropout layers are added after each LSTM layer to help prevent overfitting. The output layer uses a sigmoid activation function to produce a probability for the binary classification task. Now we can proceed to compile the model before training it.

In [ ]:
# Compile the LSTM model
model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

print("Model compiled successfully.")

The LSTM model has been successfully compiled with the Adam optimizer, binary cross-entropy loss function, and accuracy as the evaluation metric. Now we can proceed to train the model using the training data. We will train the model for 30 epochs with a batch size of 32 and use 20% of the training data for validation during training. After training, we will evaluate the model's performance on the testing set to see how well it generalizes to unseen data.

In [ ]:
# Train the LSTM model
history = model.fit(
    X_train,
    y_train,
    epochs=30,
    batch_size=32,
    validation_split=0.2,
    verbose=1
)

The training and validation accuracy and loss plots provide insights into the model's performance during training. We can observe how the accuracy and loss evolve over epochs, which can help us identify if the model is overfitting or underfitting. If the validation accuracy plateaus or starts to decrease while the training accuracy continues to increase, it may indicate overfitting. Conversely, if both training and validation accuracy are low, it may indicate underfitting. Based on these observations, we can make decisions about potential adjustments to the model architecture or training process. Now we can proceed to evaluate the model's performance on the testing set to see how well it generalizes to unseen data.

In [ ]:
import matplotlib.pyplot as plt

# Plot training and validation accuracy
plt.figure(figsize=(8, 5))
plt.plot(history.history["accuracy"], label="Training Accuracy")
plt.plot(history.history["val_accuracy"], label="Validation Accuracy")
plt.title("LSTM Model Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
# Plot training and validation loss
plt.figure(figsize=(8, 5))
plt.plot(history.history["loss"], label="Training Loss")
plt.plot(history.history["val_loss"], label="Validation Loss")
plt.title("LSTM Model Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
# Make predictions on the test dataset
y_pred_prob = model.predict(X_test)

# Convert prediction probabilities into class labels
# If probability is greater than 0.5, classify as 1
# Otherwise, classify as 0
y_pred = (y_pred_prob > 0.5).astype(int)

# Display first 10 prediction probabilities
print("First 10 prediction probabilities:")
print(y_pred_prob[:10])

# Display first 10 predicted classes
print("\nFirst 10 predicted classes:")
print(y_pred[:10])

# Display first 10 actual classes
print("\nFirst 10 actual classes:")
print(y_test[:10])

In [ ]:
# Flatten predicted labels for evaluation
y_pred = y_pred.flatten()

# Check shapes
print("y_test shape:", y_test.shape)
print("y_pred shape:", y_pred.shape)

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.metrics import confusion_matrix, classification_report

# Calculate evaluation metrics
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

print("Accuracy:", accuracy)
print("Precision:", precision)
print("Recall:", recall)
print("F1-score:", f1)

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.metrics import confusion_matrix, classification_report

# Calculate evaluation metrics
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, zero_division=0)
recall = recall_score(y_test, y_pred, zero_division=0)
f1 = f1_score(y_test, y_pred, zero_division=0)

# Print results
print("Model Evaluation Results")
print("------------------------")
print("Accuracy:", accuracy)
print("Precision:", precision)
print("Recall:", recall)
print("F1-score:", f1)

# Classification report
print("\nClassification Report:")
print(classification_report(y_test, y_pred, zero_division=0))

# Confusion matrix
cm = confusion_matrix(y_test, y_pred)

print("\nConfusion Matrix:")
print(cm)

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# Plot confusion matrix
plt.figure(figsize=(6, 4))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=["Down/Unchanged", "Up"],
    yticklabels=["Down/Unchanged", "Up"]
)

plt.title("Confusion Matrix")
plt.xlabel("Predicted Class")
plt.ylabel("Actual Class")
plt.show()

In [ ]:
# Plot actual vs predicted movement for first 100 test samples
plt.figure(figsize=(12, 5))
plt.plot(y_test[:100], label="Actual Movement", marker="o")
plt.plot(y_pred[:100], label="Predicted Movement", marker="x")

plt.title("Actual vs Predicted KLSE Movement")
plt.xlabel("Test Sample")
plt.ylabel("Movement Class")
plt.yticks([0, 1], ["Down/Unchanged", "Up"])
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
# Save the trained LSTM model
model.save("./Result/KLSE_LSTM_model.h5")

print("Model saved successfully as KLSE_LSTM_model.h5")

In [ ]:
# Save evaluation metrics into a dictionary
evaluation_results = {
    "Accuracy": accuracy,
    "Precision": precision,
    "Recall": recall,
    "F1-score": f1
}

# Convert to DataFrame
evaluation_df = pd.DataFrame([evaluation_results])

# Save as CSV
evaluation_df.to_csv("./Result/KLSE_LSTM_evaluation_results.csv", index=False)

print("Evaluation results saved successfully.")

In [ ]:
# Save actual and predicted results
prediction_results = pd.DataFrame({
    "Actual": y_test,
    "Predicted": y_pred
})

# Save as CSV
prediction_results.to_csv("./Result/KLSE_LSTM_prediction_results.csv", index=False)

print("Prediction results saved successfully.")

In [ ]:
# Save confusion matrix as CSV
confusion_matrix_df = pd.DataFrame(
    cm,
    index=["Actual Down/Unchanged", "Actual Up"],
    columns=["Predicted Down/Unchanged", "Predicted Up"]
)

confusion_matrix_df.to_csv("./Result/KLSE_LSTM_confusion_matrix.csv")

print("Confusion matrix saved successfully.")